# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and enables programmatic access to metadata and data. All entities (record sets, fields, etc.) are referenced by their `@id` according to the Croissant specification.

In [ ]:
# Install mlcroissant library if needed
!pip install -q mlcroissant

## 1. Data Loading

We begin by loading the metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)
# Metadata access is via the metadata attribute
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Explore available record sets and their fields via their `@id`s.

In [ ]:
# List all record sets in the dataset
print("Available record sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '')}")

# For each record set, print available fields (by @id)
for record_set in dataset.record_sets:
    print(f"\nRecord set '@id': {record_set['@id']} ({record_set.get('name', '')})")
    print("  Fields (by @id):")
    for field in record_set.get('field', []):
        print(f"   - {field['@id']}: {field.get('name', '')}")

## 3. Data Extraction

Let’s load data from a chosen record set (using its `@id`). Here, we select the main protein quantitation table from the dataset (usually there's a main table for record-based access).

In [ ]:
# Define main record set @id (determined from overview above)
# For the FAIR² dataset, the main record set may be: 'cr:RecordSet/1' or similar (@id). We'll retrieve the first as example.
record_set_ids = [record_set['@id'] for record_set in dataset.record_sets]

print(f"Available record_set @ids: {record_set_ids}")
main_record_set_id = None
if len(record_set_ids):
    main_record_set_id = record_set_ids[0]

# Load data from all record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records):
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

# Show column names from the main record set
if main_record_set_id:
    print(f"Columns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let’s conduct basic analysis: filtering, normalization, and grouping using selected fields by their `@id`/column.

In [ ]:
# Choose a numeric field for analysis (e.g., 'coverage_percent' if it exists)
numeric_field_candidates = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype != object]
print(f"Numeric fields found: {numeric_field_candidates}")
numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None

if numeric_field:
    threshold = 10
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a field to group by (e.g., 'accession' or similar non-numeric field)
    group_candidates = [c for c in dataframes[main_record_set_id].columns if c != numeric_field and dataframes[main_record_set_id][c].dtype == object]
    group_field = group_candidates[0] if group_candidates else None

    print(f"\nPossible group_by fields: {group_candidates}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization

Show the distribution of a selected numeric field and the average value by a group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of "{numeric_field}"')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None:
        top_groups = filtered_df[group_field].value_counts().index[:10]
        plt.figure(figsize=(9, 5))
        sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f'"{numeric_field}" by "{group_field}" (top 10 groups)")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² dataset described by a Croissant schema, explored its metadata programmatically, and performed basic filtering and visualization. To go further, link domain knowledge to field `@id`s, study experimental variables, and explore more advanced modeling or use-case-specific analyses.